In [1]:
#import os
import scanpy as sc
import anndata as ad
import pandas as pd
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
import matplotlib.pyplot as plt

In [2]:
# Load embedding
df_geneformer = pd.read_csv(
    "/projects/bioinformatics/DB/scRNAseq_parkinson/geneformer_embedding.csv",
    index_col=0
)

In [3]:
# Load metadata properly
metadata = pd.read_csv('/projects/bioinformatics/DB/scRNAseq_parkinson/metadata.csv')

In [7]:
# Step 1: create AnnData
adata = ad.AnnData(df_geneformer.values)

# Step 2: optionally give obs_names
adata.obs_names = df_geneformer.index.astype(str)  # or just leave them as 0,1,2...

# Step 3: assign metadata by position
adata.obs["disease"] = metadata["disease"].values
adata.obs["tissue"] = metadata["tissue"].values

# Step 4: combined label
adata.obs["tissue_condition"] = (
    adata.obs["tissue"].astype(str) + "_" + adata.obs["disease"].astype(str)
)

# Step 5: check
adata.obs.isna().mean()

disease             0.0
tissue              0.0
tissue_condition    0.0
dtype: float64

In [8]:
adata.obs

,disease,tissue,tissue_condition
0,Parkinson disease,medial globus pallidus,medial globus pallidus_Parkinson disease
1,Parkinson disease,primary motor cortex,primary motor cortex_Parkinson disease
2,Parkinson disease,medial globus pallidus,medial globus pallidus_Parkinson disease
3,Parkinson disease,primary motor cortex,primary motor cortex_Parkinson disease
4,Parkinson disease,primary motor cortex,primary motor cortex_Parkinson disease
...,...,...,...
2096150,normal,primary motor cortex,primary motor cortex_normal
2096151,Parkinson disease,primary visual cortex,primary visual cortex_Parkinson disease
2096152,Parkinson disease,primary motor cortex,primary motor cortex_Parkinson disease
2096153,normal,primary motor cortex,primary motor cortex_normal


In [13]:
# Build graph once
sc.pp.neighbors(adata, n_neighbors=15, use_rep='X')
adata.write("/home/znazari/scRNAseq_parkinson/result/adata_tissue_geneformer_with_neighbors.h5ad")

In [10]:
adata = sc.read('/home/znazari/scRNAseq_parkinson/result/adata_tissue_geneformer_with_neighbors.h5ad')

In [14]:
# Smart function to compute NMI
def compute_nmi_table(adata, resolutions, label_keys):
    results = []

    for res in resolutions:
        key = f"leiden_r{res}"
        sc.tl.leiden(adata, resolution=res, key_added=key, random_state=0)

        for label in label_keys:
            valid = adata.obs[[key, label]].dropna()

            nmi = normalized_mutual_info_score(
                valid[key].astype(str),
                valid[label].astype(str)
            )

            results.append({
                "resolution": res,
                "label": label,
                "NMI": nmi
            })

    return pd.DataFrame(results)


In [ ]:
# Run
resolutions = [0.3, 0.5, 0.8, 1.0]

labels_to_test = [
    "tissue",
    "disease",
    "tissue_condition"
]

nmi_results = compute_nmi_table(
    adata,
    resolutions=resolutions,
    label_keys=labels_to_test
)

print(nmi_results)


In [ ]:
# Save everything
nmi_results.to_csv("/home/znazari/scRNAseq_parkinson/result/geneformer_tissue_nmi_by_resolution.csv", index=False)
adata.write("/home/znazari/scRNAseq_parkinson/result/adata_tissue_geneformer_leiden_sweep.h5ad")


In [ ]:
nmi_result = pd.read_csv("/home/znazari/scRNAseq_parkinson/result/geneformer_tissue_nmi_by_resolution.csv")

In [ ]:
nmi_result

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Pivot for clean plotting
pivot_df = nmi_result.pivot(index="resolution", columns="label", values="NMI")

# Create figure
plt.figure(figsize=(7, 5))

# Plot each label (no explicit colors)
for col in pivot_df.columns:
    plt.plot(
        pivot_df.index,
        pivot_df[col],
        marker="o",
        linewidth=2,
        label=col
    )

# Axis labels
plt.xlabel("Leiden Resolution", fontsize=12)
plt.ylabel("Normalized Mutual Information (NMI)", fontsize=12)

# Title
plt.title("Clustering Agreement with Biological Labels Across Resolutions", fontsize=13)

# Legend
plt.legend(frameon=False, title="Label")

# Grid (subtle, journal style)
plt.grid(alpha=0.3)

# Layout
plt.tight_layout()

plt.show()
